# TimeGPT + semantic neighbours - exog_small & exog_all (h in {1, 6, 12})

Upstream source: `sem_timegpt_exog_all&small.ipynb` in the thesis workbench (the `&` was stripped from the target filename).

**Input**: `KEYWORDS_DIR_5 / *.parquet` (per-keyword panels with 5 semantic neighbours).
**Outputs**: two parallel experiment blocks, each writing its own folder:
- `timegpt_results_dir('sem_timegpt_exog_all')/forecasts/h{1,6,12}/` (all exog features)
- `timegpt_results_dir('sem_timegpt_exog_small')/forecasts/h{1,6,12}/` (only the `cpc_lastweek*` neighbour features)


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_5, timegpt_results_dir,
)
from api_keys import get_nixtla_key  # noqa: E402

ensure_env()
os.environ['NIXTLA_API_KEY'] = get_nixtla_key()


# Semantic Timegpt Exog all and Exogenous SMALL baseline calculation

In [ ]:

!pip -q install nixtla polars pyarrow tqdm scikit-learn

import os, re, warnings
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from nixtla import NixtlaClient


In [ ]:
# ==================== TimeGPT Baseline (per keyword) with 5 neighbor CPC exogs ====================
# - Excludes ['keyword', 'cpc_week', 'adclicks_sum', 'adcost_sum'] always
# - For each keyword parquet, use y = cpc_week
# - Uses exactly 5 exogenous features: columns named "cpc_lastweek__*"
# - Forecasts horizons h = [1, 6, 12], leakage-safe (fresh split each time)
# - Excludes blacklisted keywords if a blacklist file exists
# - Creates timegpt_results_dir("sem_timegpt_exog_all") (or sem_timegpt_exog_small)
# ================================================================================================

!pip -q install nixtla polars pyarrow tqdm

import os, re, warnings
from pathlib import Path
from datetime import date
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from nixtla import NixtlaClient

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('future.no_silent_downcasting', True)

# ---------------- Paths & config ----------------
BASE = DATA_ROOT
CANDIDATE_INPUT_DIRS = [
    KEYWORDS_DIR_5,
    DATA_ROOT / "keywords_dfs_full",
]
INPUT_DIR = None
for d in CANDIDATE_INPUT_DIRS:
    if d.exists() and any(d.glob("*.parquet")):
        INPUT_DIR = d
        break
assert INPUT_DIR is not None, f"No parquet files found in: {CANDIDATE_INPUT_DIRS}"

OUT_ROOT = timegpt_results_dir("sem_timegpt_exog_all")
FC_DIR_H1 = OUT_ROOT / "forecasts" / "h1"
FC_DIR_H6 = OUT_ROOT / "forecasts" / "h6"
FC_DIR_H12 = OUT_ROOT / "forecasts" / "h12"
for p in [OUT_ROOT, FC_DIR_H1, FC_DIR_H6, FC_DIR_H12]:
    p.mkdir(parents=True, exist_ok=True)

FREQ = "W-MON"
HORIZONS = [1, 6, 12]
MIN_EXTRA_WEEKS = 5

BLACKLIST_PATH = BASE / "blacklist.txt"
BLACKLIST = set()
if BLACKLIST_PATH.exists():
    BLACKLIST = {line.strip() for line in BLACKLIST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()}

# Columns always excluded from features
ALWAYS_EXCLUDE_COLS = ['keyword', 'cpc_week', 'adclicks_sum', 'adcost_sum']

# ---------------- Auth ----------------
# NIXTLA_API_KEY is set by the bootstrap cell via get_nixtla_key().

assert os.environ.get("NIXTLA_API_KEY"), "NIXTLA_API_KEY not found."
client = NixtlaClient(api_key=os.environ["NIXTLA_API_KEY"])

# ---------------- Helpers ----------------
def iso_to_date_robust(val):
    if hasattr(val, "isocalendar"):
        iso = val.isocalendar()
        return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    m1 = re.fullmatch(r"(?P<wk>\d{1,2})-(?P<yr>\d{4})", s)
    if m1:
        wk, yr = int(m1["wk"]), int(m1["yr"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    m2 = re.fullmatch(r"(?P<yr>\d{4})-(?P<wk>\d{1,2})", s)
    if m2:
        yr, wk = int(m2["yr"]), int(m2["wk"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    try:
        ts = pd.to_datetime(s, errors="coerce")
        if pd.notna(ts):
            iso = ts.isocalendar()
            return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    except Exception:
        pass
    return None

def build_weekly_grid(pdf: pd.DataFrame, freq=FREQ):
    if pdf.empty:
        return pdf
    pdf = pdf.sort_values("ds")
    pdf = pdf.groupby("ds", as_index=False).mean(numeric_only=True)
    idx = pd.date_range(pdf["ds"].min(), pdf["ds"].max(), freq=freq)
    out = pdf.set_index("ds").reindex(idx)
    out.index.name = "ds"
    return out.reset_index()

def impute_train_y_no_leak(tr: pd.DataFrame):
    tr = tr.copy().set_index("ds")
    tr["y"] = pd.to_numeric(tr["y"], errors="coerce")
    tr["y"] = tr["y"].interpolate(method="time", limit_direction="both").ffill().bfill()
    return tr.reset_index()

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    return 100 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask])

def rmse(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def pick_neighbor_cols(cols, k=5):
    valid_cols = [c for c in cols if c not in ALWAYS_EXCLUDE_COLS]
    cands = sorted([c for c in valid_cols if c.startswith("cpc_lastweek__")])
    return cands[:k]

def preprocess_keyword_file(parquet_path: Path, need_k_neighbors=5):
    try:
        df_pl = pl.read_parquet(parquet_path)
    except Exception as e:
        return None, None, ("fatal_read", str(e))
    raw_cols = df_pl.columns
    if "week" not in raw_cols or "cpc_week" not in raw_cols:
        return None, None, ("missing_cols", f"has={raw_cols}")
    neighbor_cols = pick_neighbor_cols(raw_cols, k=need_k_neighbors)
    if len(neighbor_cols) < need_k_neighbors:
        return None, None, ("insufficient_neighbors", len(neighbor_cols))
    keep_cols = ["week", "cpc_week"] + neighbor_cols
    df_pl = (
        df_pl.select(keep_cols)
             .with_columns(pl.col("week").map_elements(iso_to_date_robust, return_dtype=pl.Date).alias("ds"))
             .drop("week")
             .filter(pl.col("ds").is_not_null())
             .sort("ds")
    )
    pdf = df_pl.to_pandas().rename(columns={"cpc_week": "y"})
    pdf = build_weekly_grid(pdf, FREQ)
    if len(pdf) <= max(HORIZONS) + MIN_EXTRA_WEEKS:
        return None, None, ("too_short", len(pdf))
    return pdf, neighbor_cols, None

# ---------------- Main loop ----------------
files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Using input dir: {INPUT_DIR}")
print(f"Found {len(files)} parquet files.")

metrics_rows, skipped_rows = [], []
reasons = Counter()

for p in tqdm(files, desc="Baseline per-key (5 neighbor exogs)", leave=False):
    kw = p.stem
    if kw in BLACKLIST:
        reasons["blacklisted"] += 1
        skipped_rows.append({"keyword": kw, "reason": "blacklisted"})
        continue
    pdf, exog_cols, err = preprocess_keyword_file(p, need_k_neighbors=5)
    if pdf is None:
        reasons[err[0]] += 1
        skipped_rows.append({"keyword": kw, "reason": err[0], "info": err[1] if len(err)>1 else ""})
        continue
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    for c in exog_cols + ["y"]:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

    row = {"keyword": kw, "n_obs": len(pdf), "n_exogs": len(exog_cols)}
    for h in HORIZONS:
        split = len(pdf) - h
        tr, te = pdf.iloc[:split].copy(), pdf.iloc[split:].copy()
        if te["y"].isna().any():
            reasons[f"nan_in_test_y_h{h}"] += 1
            continue
        tr_imp = impute_train_y_no_leak(tr)
        if tr_imp["y"].isna().any():
            reasons[f"nan_remain_in_train_y_h{h}"] += 1
            continue
        if exog_cols:
            tr_imp[exog_cols] = tr_imp[exog_cols].ffill().bfill().fillna(0.0)
        df_tr_api = tr_imp[["ds", "y"] + exog_cols].copy()
        df_tr_api["unique_id"] = kw
        df_tr_api = df_tr_api[["unique_id", "ds", "y"] + exog_cols]
        try:
            fc = client.forecast(df=df_tr_api, h=h, freq=FREQ, hist_exog_list=exog_cols)
            mer = te[["ds", "y"]].copy()
            mer["unique_id"] = kw
            mer = mer.merge(fc[["unique_id", "ds", "TimeGPT"]], on=["unique_id", "ds"], how="inner")
            if not mer.empty:
                row[f"smape_h{h}"] = smape(mer["y"], mer["TimeGPT"])
                row[f"rmse_h{h}"] = rmse(mer["y"], mer["TimeGPT"])
                out_dir = {1: FC_DIR_H1, 6: FC_DIR_H6, 12: FC_DIR_H12}[h]
                mer.rename(columns={"TimeGPT": "yhat"}, inplace=True)
                mer.to_csv(out_dir / f"{kw}.csv", index=False)
        except Exception as e:
            reasons[f"api_error_h{h}"] += 1
            skipped_rows.append({"keyword": kw, "reason": f"api_error_h{h}", "info": str(e)})
    metrics_rows.append(row)

# ---------------- Save outputs ----------------
METRICS_CSV = OUT_ROOT / "metrics_baseline_exog_h1_h6_h12.csv"
SKIPPED_CSV = OUT_ROOT / "skipped_baseline_exog.csv"
metrics_df = pd.DataFrame(metrics_rows).sort_values("keyword").reset_index(drop=True)
metrics_df.to_csv(METRICS_CSV, index=False)
if skipped_rows:
    pd.DataFrame(skipped_rows).to_csv(SKIPPED_CSV, index=False)

# ---------------- Print summary ----------------
def safe_stats(a):
    a = np.array([x for x in a if pd.notna(x)], float)
    return (float(np.mean(a)), float(np.median(a))) if a.size else (np.nan, np.nan)

for h in HORIZONS:
    mu_s, md_s = safe_stats(metrics_df.get(f"smape_h{h}", []))
    mu_r, md_r = safe_stats(metrics_df.get(f"rmse_h{h}", []))
    print(f"\n==== Aggregate Metrics (h={h}) ====")
    print(f"SMAPE mean:   {mu_s:.4f}")
    print(f"SMAPE median: {md_s:.4f}")
    print(f"RMSE  mean:   {mu_r:.4f}")
    print(f"RMSE  median: {md_r:.4f}")

print("\n✅ Done (per-keyword baseline with 5 neighbor exogs; leakage-safe).")
print(f"- Metrics CSV: {METRICS_CSV}")
print(f"- Forecasts:   {OUT_ROOT / 'forecasts'}")


In [ ]:
# === Evaluation: TimeGPT baseline exog (h=1/6/12) — wide format ===
import pandas as pd
from pathlib import Path

METRICS = Path(
    timegpt_results_dir("sem_timegpt_exog_all") / "metrics_baseline_exog_h1_h6_h12.csv"
)

# Load (auto-detect delimiter)
df = pd.read_csv(METRICS, sep=None, engine="python")

# Ensure numeric (handles blanks)
for h in [1, 6, 12]:
    df[f"smape_h{h}"] = pd.to_numeric(df[f"smape_h{h}"], errors="coerce")
    df[f"rmse_h{h}"]  = pd.to_numeric(df[f"rmse_h{h}"], errors="coerce")

# Aggregate per horizon
rows = []
for h in [1, 6, 12]:
    rows.append({
        "horizon": h,
        "n_keywords": df["keyword"].nunique() if "keyword" in df.columns else df[f"smape_h{h}"].count(),
        "SMAPE_mean": df[f"smape_h{h}"].mean(),
        "SMAPE_median": df[f"smape_h{h}"].median(),
        "SMAPE_std": df[f"smape_h{h}"].std(ddof=1),
        "RMSE_mean": df[f"rmse_h{h}"].mean(),
        "RMSE_median": df[f"rmse_h{h}"].median(),
        "RMSE_std": df[f"rmse_h{h}"].std(ddof=1),
    })

summary = pd.DataFrame(rows)

# Pretty formatting
summary_display = summary.copy()
for c in [
    "SMAPE_mean", "SMAPE_median", "SMAPE_std",
    "RMSE_mean", "RMSE_median", "RMSE_std"
]:
    summary_display[c] = summary_display[c].round(4)

print("=== Aggregate Metrics (per horizon) ===")
display(summary_display)


In [ ]:
# ==================== TimeGPT Baseline (per keyword) with 5 neighbor CPC exogs ====================
# - Excludes ['keyword', 'cpc_week', 'adclicks_sum', 'adcost_sum' and the semantics] always
# - For each keyword parquet, use y = cpc_week
# - Uses exactly 5 exogenous features: columns named "cpc_lastweek__*"
# - Forecasts horizons h = [1, 6, 12], leakage-safe (fresh split each time)
# - Excludes blacklisted keywords if a blacklist file exists
# - Creates timegpt_results_dir("sem_timegpt_exog_all") (or sem_timegpt_exog_small)
# ================================================================================================

!pip -q install nixtla polars pyarrow tqdm

import os, re, warnings
from pathlib import Path
from datetime import date
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from nixtla import NixtlaClient

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('future.no_silent_downcasting', True)

# ---------------- Paths & config ----------------
BASE = DATA_ROOT
CANDIDATE_INPUT_DIRS = [
    KEYWORDS_DIR_5,
    DATA_ROOT / "keywords_dfs_full",
]
INPUT_DIR = None
for d in CANDIDATE_INPUT_DIRS:
    if d.exists() and any(d.glob("*.parquet")):
        INPUT_DIR = d
        break
assert INPUT_DIR is not None, f"No parquet files found in: {CANDIDATE_INPUT_DIRS}"

OUT_ROOT = timegpt_results_dir("sem_timegpt_exog_small")
FC_DIR_H1 = OUT_ROOT / "forecasts" / "h1"
FC_DIR_H6 = OUT_ROOT / "forecasts" / "h6"
FC_DIR_H12 = OUT_ROOT / "forecasts" / "h12"
for p in [OUT_ROOT, FC_DIR_H1, FC_DIR_H6, FC_DIR_H12]:
    p.mkdir(parents=True, exist_ok=True)

FREQ = "W-MON"
HORIZONS = [1, 6, 12]
MIN_EXTRA_WEEKS = 5

BLACKLIST_PATH = BASE / "blacklist.txt"
BLACKLIST = set()
if BLACKLIST_PATH.exists():
    BLACKLIST = {line.strip() for line in BLACKLIST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()}

# Columns always excluded from features
ALWAYS_EXCLUDE_COLS = ['keyword', 'cpc_week', 'adclicks_sum', 'adcost_sum',"avg_sim_top25_this_week", "avg_sim_top25_last_week","n_sim_this_week", "n_sim_last_week"]

# ---------------- Auth ----------------
# NIXTLA_API_KEY is set by the bootstrap cell via get_nixtla_key().

assert os.environ.get("NIXTLA_API_KEY"), "NIXTLA_API_KEY not found."
client = NixtlaClient(api_key=os.environ["NIXTLA_API_KEY"])

# ---------------- Helpers ----------------
def iso_to_date_robust(val):
    if hasattr(val, "isocalendar"):
        iso = val.isocalendar()
        return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    m1 = re.fullmatch(r"(?P<wk>\d{1,2})-(?P<yr>\d{4})", s)
    if m1:
        wk, yr = int(m1["wk"]), int(m1["yr"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    m2 = re.fullmatch(r"(?P<yr>\d{4})-(?P<wk>\d{1,2})", s)
    if m2:
        yr, wk = int(m2["yr"]), int(m2["wk"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    try:
        ts = pd.to_datetime(s, errors="coerce")
        if pd.notna(ts):
            iso = ts.isocalendar()
            return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    except Exception:
        pass
    return None

def build_weekly_grid(pdf: pd.DataFrame, freq=FREQ):
    if pdf.empty:
        return pdf
    pdf = pdf.sort_values("ds")
    pdf = pdf.groupby("ds", as_index=False).mean(numeric_only=True)
    idx = pd.date_range(pdf["ds"].min(), pdf["ds"].max(), freq=freq)
    out = pdf.set_index("ds").reindex(idx)
    out.index.name = "ds"
    return out.reset_index()

def impute_train_y_no_leak(tr: pd.DataFrame):
    tr = tr.copy().set_index("ds")
    tr["y"] = pd.to_numeric(tr["y"], errors="coerce")
    tr["y"] = tr["y"].interpolate(method="time", limit_direction="both").ffill().bfill()
    return tr.reset_index()

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    return 100 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask])

def rmse(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def pick_neighbor_cols(cols, k=5):
    valid_cols = [c for c in cols if c not in ALWAYS_EXCLUDE_COLS]
    cands = sorted([c for c in valid_cols if c.startswith("cpc_lastweek__")])
    return cands[:k]

def preprocess_keyword_file(parquet_path: Path, need_k_neighbors=5):
    try:
        df_pl = pl.read_parquet(parquet_path)
    except Exception as e:
        return None, None, ("fatal_read", str(e))
    raw_cols = df_pl.columns
    if "week" not in raw_cols or "cpc_week" not in raw_cols:
        return None, None, ("missing_cols", f"has={raw_cols}")
    neighbor_cols = pick_neighbor_cols(raw_cols, k=need_k_neighbors)
    if len(neighbor_cols) < need_k_neighbors:
        return None, None, ("insufficient_neighbors", len(neighbor_cols))
    keep_cols = ["week", "cpc_week"] + neighbor_cols
    df_pl = (
        df_pl.select(keep_cols)
             .with_columns(pl.col("week").map_elements(iso_to_date_robust, return_dtype=pl.Date).alias("ds"))
             .drop("week")
             .filter(pl.col("ds").is_not_null())
             .sort("ds")
    )
    pdf = df_pl.to_pandas().rename(columns={"cpc_week": "y"})
    pdf = build_weekly_grid(pdf, FREQ)
    if len(pdf) <= max(HORIZONS) + MIN_EXTRA_WEEKS:
        return None, None, ("too_short", len(pdf))
    return pdf, neighbor_cols, None

# ---------------- Main loop ----------------
files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Using input dir: {INPUT_DIR}")
print(f"Found {len(files)} parquet files.")

metrics_rows, skipped_rows = [], []
reasons = Counter()

for p in tqdm(files, desc="Baseline per-key (5 neighbor exogs)", leave=False):
    kw = p.stem
    if kw in BLACKLIST:
        reasons["blacklisted"] += 1
        skipped_rows.append({"keyword": kw, "reason": "blacklisted"})
        continue
    pdf, exog_cols, err = preprocess_keyword_file(p, need_k_neighbors=5)
    if pdf is None:
        reasons[err[0]] += 1
        skipped_rows.append({"keyword": kw, "reason": err[0], "info": err[1] if len(err)>1 else ""})
        continue
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    for c in exog_cols + ["y"]:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

    row = {"keyword": kw, "n_obs": len(pdf), "n_exogs": len(exog_cols)}
    for h in HORIZONS:
        split = len(pdf) - h
        tr, te = pdf.iloc[:split].copy(), pdf.iloc[split:].copy()
        if te["y"].isna().any():
            reasons[f"nan_in_test_y_h{h}"] += 1
            continue
        tr_imp = impute_train_y_no_leak(tr)
        if tr_imp["y"].isna().any():
            reasons[f"nan_remain_in_train_y_h{h}"] += 1
            continue
        if exog_cols:
            tr_imp[exog_cols] = tr_imp[exog_cols].ffill().bfill().fillna(0.0)
        df_tr_api = tr_imp[["ds", "y"] + exog_cols].copy()
        df_tr_api["unique_id"] = kw
        df_tr_api = df_tr_api[["unique_id", "ds", "y"] + exog_cols]
        try:
            fc = client.forecast(df=df_tr_api, h=h, freq=FREQ, hist_exog_list=exog_cols)
            mer = te[["ds", "y"]].copy()
            mer["unique_id"] = kw
            mer = mer.merge(fc[["unique_id", "ds", "TimeGPT"]], on=["unique_id", "ds"], how="inner")
            if not mer.empty:
                row[f"smape_h{h}"] = smape(mer["y"], mer["TimeGPT"])
                row[f"rmse_h{h}"] = rmse(mer["y"], mer["TimeGPT"])
                out_dir = {1: FC_DIR_H1, 6: FC_DIR_H6, 12: FC_DIR_H12}[h]
                mer.rename(columns={"TimeGPT": "yhat"}, inplace=True)
                mer.to_csv(out_dir / f"{kw}.csv", index=False)
        except Exception as e:
            reasons[f"api_error_h{h}"] += 1
            skipped_rows.append({"keyword": kw, "reason": f"api_error_h{h}", "info": str(e)})
    metrics_rows.append(row)

# ---------------- Save outputs ----------------
METRICS_CSV = OUT_ROOT / "metrics_baseline_exog_h1_h6_h12.csv"
SKIPPED_CSV = OUT_ROOT / "skipped_baseline_exog.csv"
metrics_df = pd.DataFrame(metrics_rows).sort_values("keyword").reset_index(drop=True)
metrics_df.to_csv(METRICS_CSV, index=False)
if skipped_rows:
    pd.DataFrame(skipped_rows).to_csv(SKIPPED_CSV, index=False)

# ---------------- Print summary ----------------
def safe_stats(a):
    a = np.array([x for x in a if pd.notna(x)], float)
    return (float(np.mean(a)), float(np.median(a))) if a.size else (np.nan, np.nan)

for h in HORIZONS:
    mu_s, md_s = safe_stats(metrics_df.get(f"smape_h{h}", []))
    mu_r, md_r = safe_stats(metrics_df.get(f"rmse_h{h}", []))
    print(f"\n==== Aggregate Metrics (h={h}) ====")
    print(f"SMAPE mean:   {mu_s:.4f}")
    print(f"SMAPE median: {md_s:.4f}")
    print(f"RMSE  mean:   {mu_r:.4f}")
    print(f"RMSE  median: {md_r:.4f}")

print("\n✅ Done (per-keyword baseline with 5 neighbor exogs; leakage-safe).")
print(f"- Metrics CSV: {METRICS_CSV}")
print(f"- Forecasts:   {OUT_ROOT / 'forecasts'}")
